# Description Semantic GAT Inputs

This notebook rebuilds the same three GAT input structures produced by the original bike_prepare_data/bike_gat.ipynb, but swaps in the already generated description embeddings. The outputs are desc_graph_df_minilm.parquet, desc_graph_gpt.parquet, and desc_graph_gpt_small.parquet in data/processed/model_df/desc_semantics.


In [ ]:
# =========================================================
# 1. Imports and path configuration
# This cell defines every input and output path used by the desc semantic GAT prep.
# The output names intentionally mirror the original GAT files with desc_ added.
# No embedding model is called here; all embeddings are read from the parquet files already produced by wiki_desc_prep.
# =========================================================

from pathlib import Path
import numpy as np
import pandas as pd

PROJECT_ROOT = Path('/home/najla/dev/najla-msc')
BIKESHARE_ROOT = PROJECT_ROOT / 'bikeshare'
MODEL_DF_DIR = BIKESHARE_ROOT / 'data/processed/model_df'
GRAPH_DIR = BIKESHARE_ROOT / 'data/processed/graph'
DESC_EMBEDDING_DIR = GRAPH_DIR / 'desc_semantic'
DESC_OUTPUT_DIR = MODEL_DF_DIR / 'desc_semantics'
DESC_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

BASE_MODEL_FILE = MODEL_DF_DIR / 'df_2targets_txtTokens_graphFeat.parquet'
NODES_FILE = GRAPH_DIR / 'nodes.parquet'
DESC_MINILM_EMBEDDING_FILE = DESC_EMBEDDING_DIR / 'desc_all_wikidata_minilm_embeddings.parquet'
DESC_GPT_LARGE_EMBEDDING_FILE = DESC_EMBEDDING_DIR / 'desc_all_wikidata_gpt_large_embeddings.parquet'
DESC_GPT_SMALL_EMBEDDING_FILE = DESC_EMBEDDING_DIR / 'desc_all_wikidata_gpt_small_embeddings.parquet'

DESC_GRAPH_DF_MINILM_FILE = DESC_OUTPUT_DIR / 'desc_graph_df_minilm.parquet'
DESC_GRAPH_GPT_FILE = DESC_OUTPUT_DIR / 'desc_graph_gpt.parquet'
DESC_GRAPH_GPT_SMALL_FILE = DESC_OUTPUT_DIR / 'desc_graph_gpt_small.parquet'

REFERENCE_GRAPH_DF_MINILM_FILE = MODEL_DF_DIR / 'graph_df_minilm.parquet'
REFERENCE_GRAPH_GPT_FILE = MODEL_DF_DIR / 'graph_gpt.parquet'
REFERENCE_GRAPH_GPT_SMALL_FILE = MODEL_DF_DIR / 'graph_gpt_small.parquet'
BROKEN_ISLAND_LOC_ID = '5350002.00'


In [ ]:
# =========================================================
# 2. Small reusable helpers
# as_loc_id keeps loc_id formatting identical to the original bike_gat notebook.
# load_desc_embedding keeps one node-level embedding row per loc_id and optionally keeps MiniLM metadata.
# reorder_like_reference is the guardrail that forces the desc outputs to match the original graph_df_minilm, graph_gpt, and graph_gpt_small schemas exactly.
# =========================================================

def as_loc_id(series):
    return pd.to_numeric(series, errors='raise').map(lambda value: f'{value:.2f}')


def embedding_columns(df, prefix):
    return [col for col in df.columns if col.startswith(prefix)]


def load_desc_embedding(path, prefix, keep_metadata=False):
    df = pd.read_parquet(path).copy()
    df['loc_id'] = as_loc_id(df['loc_id'])
    emb_cols = embedding_columns(df, prefix)
    if not emb_cols:
        raise ValueError(f'No embedding columns found with prefix {prefix} in {path}')
    keep_cols = ['loc_id'] + emb_cols
    if keep_metadata:
        metadata_cols = [col for col in ['attraction_count', 'attraction_missing_flag'] if col in df.columns]
        keep_cols = ['loc_id'] + metadata_cols + emb_cols
    out = df[keep_cols].drop_duplicates(subset=['loc_id'], keep='first').reset_index(drop=True)
    if out['loc_id'].duplicated().any():
        raise ValueError(f'Duplicate loc_id values remain in {path}')
    return out


def reorder_like_reference(df, reference_file):
    reference_cols = list(pd.read_parquet(reference_file).columns)
    missing = [col for col in reference_cols if col not in df.columns]
    extra = [col for col in df.columns if col not in reference_cols]
    if missing:
        raise ValueError(f'Missing columns required by {reference_file.name}: {missing[:20]}')
    if extra:
        print(f'Dropping extra columns not in {reference_file.name}: {extra[:20]}')
        df = df.drop(columns=extra)
    return df[reference_cols]


def save_and_report(df, path, prefix):
    emb_cols = embedding_columns(df, prefix)
    df.to_parquet(path, index=False)
    print('=' * 72)
    print('Saved:', path)
    print('Shape:', df.shape)
    print('Embedding columns:', len(emb_cols), 'prefix:', prefix)
    print('loc_id count:', df['loc_id'].nunique())


In [ ]:
# =========================================================
# 3. Rebuild the original full CT-date graph panel
# This is the same structural step as the original bike_prepare_data/bike_gat.ipynb.
# Labelled station/date rows are kept, graph-only CT/date rows are added, and the disconnected island node is removed.
# The added graph-only rows keep NaN targets so GAT can pass messages through the full graph without using them in supervised loss.
# =========================================================

def build_full_node_df():
    df_txt_node2vec = pd.read_parquet(BASE_MODEL_FILE).copy()
    node2vec_cols = [col for col in df_txt_node2vec.columns if col.startswith('node2vec_')]
    df_txt_node2vec = df_txt_node2vec.drop(columns=node2vec_cols + ['lat', 'lon'], errors='ignore')
    df_txt_node2vec['date'] = pd.to_datetime(df_txt_node2vec['date'])
    df_txt_node2vec['loc_id'] = as_loc_id(df_txt_node2vec['loc_id'])

    all_node_features = pd.read_parquet(NODES_FILE).drop(
        columns=['loc_name', 'type', 'lon', 'lat', 'geometry', 'original_geometry'],
        errors='ignore',
    ).reset_index()
    all_node_features['loc_id'] = as_loc_id(all_node_features['loc_id'])

    temporal_cols = ['date', 'day_type', 'public_holiday', 'Main_Weather_Category', 'season']
    df_temporal_features = df_txt_node2vec[temporal_cols].drop_duplicates().reset_index(drop=True)

    df_txt_node2vec = df_txt_node2vec.loc[~df_txt_node2vec['loc_id'].eq(BROKEN_ISLAND_LOC_ID)].reset_index(drop=True)
    all_node_features = all_node_features.loc[~all_node_features['loc_id'].eq(BROKEN_ISLAND_LOC_ID)].reset_index(drop=True)

    all_node_date_df = all_node_features.merge(df_temporal_features, how='cross')
    labelled_node_date_keys = df_txt_node2vec[['loc_id', 'date']].drop_duplicates().assign(has_labelled_node_date=1)
    missing_node_date_df = (
        all_node_date_df.merge(labelled_node_date_keys, on=['loc_id', 'date'], how='left')
        .loc[lambda df: df['has_labelled_node_date'].isna()]
        .drop(columns=['has_labelled_node_date'])
        .reset_index(drop=True)
    )

    missing_node_date_df['inflow_count'] = np.nan
    missing_node_date_df['outflow_count'] = np.nan
    missing_node_date_df['spatial_group'] = np.nan
    missing_node_date_df['wiki_items_text'] = 'no_wikidata'
    missing_node_date_df['attraction_missing_flag'] = 1
    missing_node_date_df['attraction_count'] = -1
    df_txt_node2vec['wiki_items_text'] = df_txt_node2vec['wiki_items_text'].fillna('no_wikidata').replace({'no_wiki_data': 'no_wikidata'})
    missing_attraction = df_txt_node2vec['attraction_count'].isna()
    sentinel_attraction = df_txt_node2vec['attraction_count'].eq(-1)
    missing_text = df_txt_node2vec['wiki_items_text'].eq('no_wikidata')
    combined_missing = np.logical_or.reduce([missing_attraction, sentinel_attraction, missing_text])
    df_txt_node2vec['attraction_missing_flag'] = combined_missing.astype(int)
    df_txt_node2vec.loc[df_txt_node2vec['attraction_missing_flag'].eq(1), 'attraction_count'] = -1

    full_node_df = pd.concat([df_txt_node2vec, missing_node_date_df], ignore_index=True, sort=False)
    print('Full rows:', len(full_node_df))
    print('Full nodes:', full_node_df['loc_id'].nunique())
    print('Dates:', full_node_df['date'].nunique())
    print('Rows with inflow counts:', full_node_df['inflow_count'].notna().sum())
    print('Rows without outflow counts:', full_node_df['outflow_count'].isna().sum())
    return full_node_df


In [ ]:
# =========================================================
# 4. Merge desc embeddings and save the three GAT input files
# MiniLM becomes the full station-date graph dataframe desc_graph_df_minilm.parquet.
# GPT large and GPT small keep the original node-level graph_gpt and graph_gpt_small structures.
# The final schema check fails if any desc output does not match the original counterpart.
# =========================================================


full_node_df = build_full_node_df()
node_order_df = full_node_df[['loc_id']].drop_duplicates(subset=['loc_id']).reset_index(drop=True)

desc_minilm_df = node_order_df.merge(
        load_desc_embedding(DESC_MINILM_EMBEDDING_FILE, 'wiki_emb_', keep_metadata=True),
        on='loc_id',
        how='left',
    )
desc_gpt_large_df = node_order_df.merge(
        load_desc_embedding(DESC_GPT_LARGE_EMBEDDING_FILE, 'gpt_emb_', keep_metadata=False),
        on='loc_id',
        how='left',
    )
desc_gpt_small_df = node_order_df.merge(
        load_desc_embedding(DESC_GPT_SMALL_EMBEDDING_FILE, 'gpt_small_emb_', keep_metadata=False),
        on='loc_id',
        how='left',
    )

for name, df, prefix in [
        ('MiniLM', desc_minilm_df, 'wiki_emb_'),
        ('GPT large', desc_gpt_large_df, 'gpt_emb_'),
        ('GPT small', desc_gpt_small_df, 'gpt_small_emb_'),
    ]:
        emb_cols = embedding_columns(df, prefix)
        rows_with_missing = int(df[emb_cols].isna().any(axis=1).sum())
        print(name, 'shape:', df.shape, 'embedding_cols:', len(emb_cols), 'rows_with_missing_embeddings:', rows_with_missing)
        if rows_with_missing:
            missing_ids = df.loc[df[emb_cols].isna().any(axis=1), 'loc_id'].head(20).tolist()
            raise ValueError(f'{name} has missing embeddings for loc_id values: {missing_ids}')

old_embedding_cols = [
        col for col in full_node_df.columns
        if col.startswith(('wiki_emb_', 'gpt_emb_', 'gpt_small_emb_'))
    ]
base_full_node_df = full_node_df.drop(
        columns=old_embedding_cols + ['wiki_items_text', 'token_attraction_count', 'attraction_count', 'attraction_missing_flag'],
        errors='ignore',
    )

desc_graph_df_minilm = base_full_node_df.merge(desc_minilm_df, on='loc_id', how='left')
desc_graph_df_minilm = reorder_like_reference(desc_graph_df_minilm, REFERENCE_GRAPH_DF_MINILM_FILE)
desc_graph_gpt = reorder_like_reference(desc_gpt_large_df, REFERENCE_GRAPH_GPT_FILE)
desc_graph_gpt_small = reorder_like_reference(desc_gpt_small_df, REFERENCE_GRAPH_GPT_SMALL_FILE)

save_and_report(desc_graph_df_minilm, DESC_GRAPH_DF_MINILM_FILE, 'wiki_emb_')
save_and_report(desc_graph_gpt, DESC_GRAPH_GPT_FILE, 'gpt_emb_')
save_and_report(desc_graph_gpt_small, DESC_GRAPH_GPT_SMALL_FILE, 'gpt_small_emb_')

for output_file, reference_file in [
        (DESC_GRAPH_DF_MINILM_FILE, REFERENCE_GRAPH_DF_MINILM_FILE),
        (DESC_GRAPH_GPT_FILE, REFERENCE_GRAPH_GPT_FILE),
        (DESC_GRAPH_GPT_SMALL_FILE, REFERENCE_GRAPH_GPT_SMALL_FILE),
    ]:
        output_cols = pd.read_parquet(output_file).columns.tolist()
        reference_cols = pd.read_parquet(reference_file).columns.tolist()
        same_columns = output_cols == reference_cols
        print(output_file.name, 'same columns as', reference_file.name + ':', same_columns)
        if not same_columns:
            raise ValueError(f'{output_file.name} does not match {reference_file.name}')


In [ ]:
desc_graph_df_minilm.head(3)


In [ ]:
desc_graph_df_minilm.shape


In [ ]:
desc_graph_df_minilm.columns


In [ ]:
df = desc_graph_df_minilm.copy()

# Exact column names in bike_gat output
LOC_ID_COL = "loc_id"
INFLOW_COL = "inflow_count"
OUTFLOW_COL = "outflow_count"
WIKI_EMB_PREFIX = "wiki_emb_"
ATTRACTION_COL = "attraction_count"
MISSING_FLAG_COL = "attraction_missing_flag"

wiki_emb_cols = [c for c in df.columns if c.startswith(WIKI_EMB_PREFIX)]

labeled_mask = df[[INFLOW_COL, OUTFLOW_COL]].notna().any(axis=1)
unlabeled_mask = df[[INFLOW_COL, OUTFLOW_COL]].isna().all(axis=1)

exclude_from_null_check = (
    [INFLOW_COL, OUTFLOW_COL, ATTRACTION_COL, MISSING_FLAG_COL]
    + wiki_emb_cols
)

null_check_cols = [
    c for c in df.columns
    if c not in exclude_from_null_check
]

summary = {
    "distinct_loc_id": df[LOC_ID_COL].nunique(),
    "rows_labeled_inflow_or_outflow": labeled_mask.sum(),
    "rows_unlabeled_inflow_and_outflow_null": unlabeled_mask.sum(),
    "rows_wiki_embeddings_zero_vector": (df[wiki_emb_cols].fillna(0).eq(0).all(axis=1)).sum(),
    "rows_attraction_count_zero": df[ATTRACTION_COL].eq(0).sum(),
    "rows_attraction_missing_flag_minus_1": df[MISSING_FLAG_COL].eq(1).sum(),
}

null_counts_rest_columns = (
    df[null_check_cols]
    .isna()
    .sum()
    .loc[lambda s: s > 0]
    .sort_values(ascending=False)
)

print("Summary:")
for k, v in summary.items():
    print(f"{k}: {v}")

print("
Null rows per remaining column:")
print(null_counts_rest_columns)


In [ ]:
rest_cols = [c for c in df.columns if c not in exclude_from_null_check]
null_counts_all_rest_columns = df[rest_cols].isna().sum().sort_values(ascending=False)

print("
Null rows per remaining column, including 0-null columns:")
print(null_counts_all_rest_columns.to_string())
